In [1]:
import argparse
import pickle
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
 
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

In [2]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [12]:
dt = np.load("curated_airfoils.npz", allow_pickle=True)


In [30]:
#If we want to go the parsing route
parser = argparse.ArgumentParser(description="Airfoil Surrogate Modeling")
parser.add_argument("--data", default="curated_airfoils.npz",
                    help="Path to curated_airfoils.npz")
parser.add_argument("--n_airfoils", type=int, default=2000,
                    help="Number of airfoils to use for training (default 2000)")
parser.add_argument("--seed", type=int, default=42)
args = parser.parse_known_args()

In [3]:
N_AIRFOILS     = 2000    # airfoils used for training (max 19164)
GPR_SUBSAMPLE  = 1500   # GPR training points (O(n^3) — keep <= 3000)
PINN_EPOCHS    = 200    # PINN training epochs
BNN_EPOCHS     = 200    # BNN training epochs
BNN_MC_SAMPLES = 50     # MC-Dropout inference samples
SEED           = 42
 
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
C1, C2, C3, C4, C5 = "#2196F3", "#F44336", "#4CAF50", "#FF9800", "#9C27B0"
CMAP_DARK  = "#0d1117"
CMAP_PANEL = "#161b22"

In [4]:
shapes = np.load('curated_airfoils.npz')['shapes']
classes = np.load('curated_airfoils.npz')['classes']
print("shapes: ", shapes.shape)
print('classes ', classes.shape)

shapes:  (19164, 1001, 2)
classes  (19164,)


In [5]:
def extract_features(shape: np.ndarray):
    """
    Return 18 geometric features from a (1001, 2) airfoil coordinate array.
 
    Features
    --------
    [0]  max_thickness       maximum thickness / chord
    [1]  x_max_thickness     chord-wise location of max thickness
    [2]  max_camber          maximum |camber| / chord
    [3]  x_max_camber        chord-wise location of max camber
    [4]  mean_camber         mean camber / chord
    [5]  te_thickness        trailing-edge thickness
    [6]  le_thickness        leading-edge thickness proxy (~1% chord)
    [7]  le_camber_slope     d(camber)/dx at leading edge
    [8]  te_camber_slope     d(camber)/dx at trailing edge
    [9-14]  thickness at x = 12, 25, 40, 60, 75, 90 %
    [15-17] camber   at x = 25, 50, 75 %
    """
    x, y   = shape[:, 0], shape[:, 1]
    le_idx = int(np.argmin(x))
 
    x_up = x[:le_idx + 1][::-1]
    y_up = y[:le_idx + 1][::-1]
    x_lo = x[le_idx:]
    y_lo = y[le_idx:]
 
    xc = np.linspace(0, 1, 200)
    try:
        yu = np.interp(xc, x_up, y_up)
        yl = np.interp(xc, x_lo, y_lo)
    except Exception:
        return None
 
    thick  = yu - yl
    camber = (yu + yl) / 2.0
 
    max_t   = float(np.max(thick))
    x_max_t = float(xc[np.argmax(thick)])
    max_c   = float(np.max(np.abs(camber)))
    x_max_c = float(xc[np.argmax(np.abs(camber))]) if max_c > 1e-8 else 0.5
    mean_c  = float(np.mean(camber))
    te_t    = float(thick[-1])
    le_t    = float(thick[2])
 
    dcamber  = np.gradient(camber, xc)
    le_slope = float(dcamber[2])
    te_slope = float(dcamber[-3])
 
    t_st = np.interp([0.12, 0.25, 0.40, 0.60, 0.75, 0.90], xc, thick)
    c_st = np.interp([0.25, 0.50, 0.75],                    xc, camber)
 
    feat = np.array([max_t, x_max_t, max_c, x_max_c, mean_c,
                     te_t, le_t, le_slope, te_slope, *t_st, *c_st])
    if np.any(np.isnan(feat)) or np.any(np.isinf(feat)):
        return None
    return feat

In [6]:
FEATURE_NAMES = [
    "max_thickness", "x_max_thickness", "max_camber", "x_max_camber",
    "mean_camber", "te_thickness", "le_thickness",
    "le_camber_slope", "te_camber_slope",
    "t@12%", "t@25%", "t@40%", "t@60%", "t@75%", "t@90%",
    "c@25%", "c@50%", "c@75%",
]
 
print("Extracting geometric features ...")
features_list, valid_idx = [], []
for i, shape in enumerate(shapes):
    f = extract_features(shape)
    if f is not None:
        features_list.append(f)
        valid_idx.append(i)
        
features   = np.array(features_list)
valid_idx  = np.array(valid_idx)
valid_cls  = classes[valid_idx]
print(f"  Valid airfoils : {len(features):,}")
 
max_t  = features[:, 0]
max_c  = features[:, 2]
mean_c = features[:, 4]

Extracting geometric features ...
  Valid airfoils : 19,164


PHYSICS-BASED Cl / Cd GENERATION
Cl - thin airfoil theory + Helmbold thickness correction + Kirchhoff flow-separation stall model
Cd - empirical drag polar  Cd = Cd0 + K*Cl^2 + post-stall drag surge

In [7]:
print("Generating physics-based Cl / Cd ...")
 
alpha_deg = np.linspace(-10, 25, 36)
 
alpha_L0     = -(2.0 * mean_c + 0.5 * features[:, 15] * 0.5)
thick_corr   = 1.0 + 0.77 * max_t
Cl_alpha_slp = 2.0 * np.pi * thick_corr
alpha_stall  = np.clip(8 + 20 * features[:, 6] + 5 * max_t - 3 * max_c, 8, 18)
Cl_max_arr   = 1.0 + 2.0 * max_c + 0.3 * max_t
Cd0          = np.clip(0.004 + 0.008 * max_t + 0.01 * max_t**2
                       + 0.5 * features[:, 5]**2, 0.003, 0.05)
K_arr        = 0.012 + 0.03 * max_c + 0.01 * np.abs(features[:, 8])

Generating physics-based Cl / Cd ...


In [8]:
def kirchhoff_cl(alpha_d, stall_d, Cl_max, Cl_alpha, al0):
    ar     = np.deg2rad(alpha_d)
    asr    = np.deg2rad(stall_d)
    Cl_lin = Cl_alpha * (ar - al0)
    delta  = ar - asr
    f      = np.where(delta <= 0, 1.0,
                      np.exp(-2.5 * delta / (0.1 + 0.3 * asr)))
    f      = np.clip(f, 0, 1)
    Cl_k   = Cl_max * ((1 + np.sqrt(f)) / 2) ** 2
    blend  = np.where(delta <= 0, 1.0, np.exp(-4.0 * np.clip(delta, 0, None)))
    Cl     = blend * Cl_lin + (1 - blend) * Cl_k
    f_neg  = np.exp(-2.5 * np.abs(np.minimum(delta, 0)) / (0.1 + 0.3 * asr))
    Cl_neg = -Cl_max * ((1 + np.sqrt(np.clip(f_neg, 0, 1))) / 2) ** 2
    return np.where(alpha_d < -stall_d, Cl_neg, Cl)
 
def compute_cd(alpha_d, stall_d, Cd0_i, K_i, Cl_arr):
    delta    = alpha_d - stall_d
    Cd_polar = Cd0_i + K_i * Cl_arr**2
    surge    = 0.8 * (1 - np.exp(-np.maximum(0, delta) / 3)) \
               * np.sin(np.deg2rad(np.abs(alpha_d)))**2
    return np.clip(Cd_polar + surge, 0.003, 3.0)

In [9]:
n_air = len(features)
n_aoa = len(alpha_deg)
Cl_all = np.zeros((n_air, n_aoa))
Cd_all = np.zeros((n_air, n_aoa))
 
for i in range(n_air):
    Cl_all[i] = kirchhoff_cl(alpha_deg, alpha_stall[i],
                              Cl_max_arr[i], Cl_alpha_slp[i], alpha_L0[i])
    Cd_all[i] = compute_cd(alpha_deg, alpha_stall[i],
                            Cd0[i], K_arr[i], Cl_all[i])
LD_all = Cl_all / np.clip(Cd_all, 1e-4, None)
print(f"  Cl in [{Cl_all.min():.3f}, {Cl_all.max():.3f}]")
print(f"  Cd in [{Cd_all.min():.4f}, {Cd_all.max():.4f}]")

  Cl in [-1.083, 2.193]
  Cd in [0.0040, 0.1892]


Build the Dataset

In [12]:
sel = rng.choice(n_air, min(N_AIRFOILS, n_air), replace=False)
 
X_geo      = np.repeat(features[sel], n_aoa, axis=0)
X_alpha    = np.tile(alpha_deg, len(sel))[:, None]
X          = np.hstack([X_geo, X_alpha])
y_Cl       = Cl_all[sel].ravel()
y_Cd       = Cd_all[sel].ravel()
alpha_flat = np.tile(alpha_deg, len(sel))
 
n_total = len(X)
n_train = int(0.8 * n_total)
perm    = rng.permutation(n_total)
tr, te  = perm[:n_train], perm[n_train:]
 
X_train, X_test     = X[tr],    X[te]
yCl_train, yCl_test = y_Cl[tr], y_Cl[te]
yCd_train, yCd_test = y_Cd[tr], y_Cd[te]
alpha_test   = alpha_flat[te]

In [13]:
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_te_s = scaler.transform(X_test)
 
high  = np.abs(alpha_test) >= 10
stall = np.abs(alpha_test) >= 15

Output Results

In [14]:
def regime_metrics(y_true, y_pred, mask):
    return {"R2":  r2_score(y_true[mask], y_pred[mask]),
            "MAE": mean_absolute_error(y_true[mask], y_pred[mask])}

def print_metrics(name, Cl_p, Cd_p):
    all_m = np.ones(len(yCl_test), bool)
    print(f"\n  [{name}]")
    print(f"  {'Regime':<14} {'Cl R2':>9} {'Cl MAE':>9} {'Cd R2':>9} {'Cd MAE':>9}")
    print(f"  {'-'*52}")
    for label, mask in [("All AoA", all_m), ("|a|>=10", high), ("|a|>=15", stall)]:
        mCl = regime_metrics(yCl_test, Cl_p, mask)
        mCd = regime_metrics(yCd_test, Cd_p, mask)
        print(f"  {label:<14} {mCl['R2']:>9.4f} {mCl['MAE']:>9.4f}"
              f" {mCd['R2']:>9.4f} {mCd['MAE']:>9.6f}")

all_preds = {}   # name -> (Cl_pred, Cd_pred)

Models

Model 1 - MLP Neural Network

In [15]:
mlp_kw = dict(hidden_layer_sizes=(128, 64, 32), activation="relu",
              max_iter=300, random_state=SEED,
              early_stopping=True, n_iter_no_change=15,
              validation_fraction=0.1)
 
mlp_Cl = MLPRegressor(**mlp_kw)
mlp_Cl.fit(X_tr_s, yCl_train)
Cl_pred_mlp = mlp_Cl.predict(X_te_s)
 
mlp_Cd = MLPRegressor(**mlp_kw)
mlp_Cd.fit(X_tr_s, yCd_train)
Cd_pred_mlp = mlp_Cd.predict(X_te_s)

In [16]:
print_metrics("MLP", Cl_pred_mlp, Cd_pred_mlp)


  [MLP]
  Regime             Cl R2    Cl MAE     Cd R2    Cd MAE
  ----------------------------------------------------
  All AoA           0.9998    0.0055    0.9998  0.000403
  |a|>=10           0.9996    0.0048    0.9999  0.000351
  |a|>=15           0.9946    0.0048    0.9997  0.000360


In [17]:
all_preds["MLP"] = (Cl_pred_mlp, Cd_pred_mlp)

Model 2 - Gaussian Process Regression (with uncertainty)